# Notebook 09 — Video Indexing

Phase 4. Video is **images × time + audio**. The architecture (S3 §8.1) most production systems use is *frame sampling + a separate audio track*, indexed independently and queried together.

We're going to build a **dual-index** search engine over a real lecture video — one signal from sampled frames (CLIP), one from the transcribed audio (Whisper + multilingual text embeddings) — and fuse them with **Reciprocal Rank Fusion (RRF)**. Same query, two retrieval routes, fused ranking. This is the pattern that powers most non-Gemini video-search products in 2025-2026.

## What this notebook covers

1. Download a short video from YouTube or Bilibili using `yt-dlp`.
2. Extract sampled frames + audio with `ffmpeg`.
3. Index frames with CLIP (one vector per frame).
4. Transcribe audio with Whisper, embed segments with a multilingual text model.
5. Search each modality alone, then fuse with RRF.
6. Test with a code-switched query — the kind of bilingual technical question your students would actually ask.

## Pick your video

Drop a short (2–10 min) lecture or talk video at `../data/video_samples/lecture.mp4`. Recommended sources:

**English (YouTube):**
```bash
yt-dlp -f 'best[height<=480]' \\
  -o '../data/video_samples/lecture.%(ext)s' \\
  'https://www.youtube.com/watch?v=YOUR_VIDEO_ID'
# Try a short Karpathy clip, or a 3Blue1Brown chapter
```

**Chinese (Bilibili):**
```bash
yt-dlp -f 'best[height<=480]' \\
  -o '../data/video_samples/lecture.%(ext)s' \\
  'https://www.bilibili.com/video/BVxxxxxxxx'
# Pick a 技术分享 from up主 like CodeSheep or 何同学
```

**For a code-switching demo specifically** — find a Chinese speaker giving a tech talk that uses English jargon throughout. Most B站 tech talks fit. That's the realistic input where the dual-index approach earns its keep.

In [ ]:
from pathlib import Path
import subprocess, json, shutil

VIDEO_DIR = Path("../data/video_samples")
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

videos = sorted(VIDEO_DIR.glob("lecture.*"))
assert videos, f"Drop a video at {VIDEO_DIR.resolve()}/lecture.mp4 (see commands above)"
VIDEO_PATH = videos[0]
assert shutil.which("ffmpeg"), "ffmpeg not in PATH — `brew install ffmpeg` (macOS) or apt install ffmpeg"

# Probe duration
duration = float(subprocess.check_output([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(VIDEO_PATH),
]).strip())
print(f"video: {VIDEO_PATH.name}  ({duration:.1f}s)")

## 1. Extract frames (1 fps) and audio

1 frame per second is a sane default for lecture content. For high-action video (sports, dance) you'd want more — see §8.1 (F-16 outperforms GPT-4o on high-speed sports tasks). The audio track is extracted as 16 kHz mono WAV — what Whisper expects.

In [ ]:
FRAMES_DIR = VIDEO_DIR / "frames"
AUDIO_PATH = VIDEO_DIR / "audio.wav"
FRAMES_DIR.mkdir(exist_ok=True)
for f in FRAMES_DIR.glob("*.jpg"):
    f.unlink()

# Extract 1 frame per second — quality scale 4 keeps file size reasonable
subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", str(VIDEO_PATH),
    "-vf", "fps=1", "-q:v", "4",
    str(FRAMES_DIR / "frame_%05d.jpg"),
], check=True)

subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", str(VIDEO_PATH),
    "-ac", "1", "-ar", "16000",   # mono, 16 kHz
    str(AUDIO_PATH),
], check=True)

frame_paths = sorted(FRAMES_DIR.glob("frame_*.jpg"))
print(f"frames: {len(frame_paths)}   audio: {AUDIO_PATH.stat().st_size / 1e6:.1f} MB")

## 2. Frame index — CLIP embeddings

Each frame's filename encodes its timestamp (`frame_00012.jpg` = second 12). CLIP gives a 512-d vector per frame. For a 5-minute video at 1 fps that's 300 vectors — trivially fast in numpy.

In [ ]:
import torch, torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor
from PIL import Image

CLIP_NAME = "openai/clip-vit-base-patch32"
device = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(device).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_NAME)

@torch.no_grad()
def clip_image_vecs(paths, batch=16):
    out = []
    for i in range(0, len(paths), batch):
        imgs = [Image.open(p).convert("RGB") for p in paths[i : i + batch]]
        inputs = clip_proc(images=imgs, return_tensors="pt").to(device)
        feats = clip_model.get_image_features(**inputs)
        out.append(F.normalize(feats, dim=-1).cpu())
    return torch.cat(out, dim=0)

@torch.no_grad()
def clip_text_vec(text):
    inputs = clip_proc(text=[text], return_tensors="pt", padding=True).to(device)
    return F.normalize(clip_model.get_text_features(**inputs), dim=-1).cpu()[0]

frame_vecs = clip_image_vecs(frame_paths)
frame_seconds = [int(p.stem.split("_")[1]) for p in frame_paths]
print(f"frame_vecs shape: {frame_vecs.shape}")

## 3. Audio index — Whisper segments + multilingual text embeddings

We use the strategy mix from nb07: VAD chunking + force-language with glossary biasing for the typical "Chinese speaker, English jargon" tech-talk profile. Adjust `language=` if your video is English-only.

In [ ]:
from faster_whisper import WhisperModel

asr = WhisperModel("large-v3", device="cpu", compute_type="int8")

# If you know the dominant language, pin it. Otherwise let Whisper auto-detect.
DOMINANT_LANG = None   # try "zh", "en", or None
GLOSSARY = (
    "attention, transformer, embedding, self-attention, OKR, function, gradient, "
    "backprop, ReLU, softmax, neural network, model, training, loss."
)

segments_gen, info = asr.transcribe(
    str(AUDIO_PATH),
    vad_filter=True,
    language=DOMINANT_LANG,
    initial_prompt=GLOSSARY if DOMINANT_LANG == "zh" else None,
)
audio_segments = list(segments_gen)
print(f"detected language: {info.language}  segments: {len(audio_segments)}")
for s in audio_segments[:5]:
    print(f"  [{s.start:6.1f} → {s.end:6.1f}]  {s.text.strip()[:80]}")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

ST_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
st = SentenceTransformer(ST_NAME)

audio_texts = [s.text.strip() for s in audio_segments]
audio_vecs = st.encode(audio_texts, normalize_embeddings=True, show_progress_bar=False)
audio_starts = [s.start for s in audio_segments]
audio_ends = [s.end for s in audio_segments]
print(f"audio_vecs shape: {audio_vecs.shape}")

## 4. Single-modality search

Each search returns a list of (timestamp_sec, score) — the timestamp is what the user wants for navigation.

In [ ]:
def search_frames(query: str, k: int = 5):
    q = clip_text_vec(query)                              # (512,)
    sims = (frame_vecs @ q).numpy()                       # (N_frames,)
    idx = np.argsort(-sims)[:k]
    return [(frame_seconds[i], float(sims[i]), "frame") for i in idx]

def search_audio(query: str, k: int = 5):
    q = st.encode([query], normalize_embeddings=True)[0]   # (384,)
    sims = audio_vecs @ q                                  # (N_segments,)
    idx = np.argsort(-sims)[:k]
    return [(audio_starts[i], float(sims[i]), "audio", audio_texts[i]) for i in idx]

# Tune these to your video
DEMO = "introduction"
print(f"\n[frame] '{DEMO}':")
for ts, sc, _ in search_frames(DEMO, k=3):
    print(f"  {ts:>5.0f}s  score={sc:.3f}")
print(f"\n[audio] '{DEMO}':")
for ts, sc, _, txt in search_audio(DEMO, k=3):
    print(f"  {ts:>5.0f}s  score={sc:.3f}  {txt[:80]}")

## 5. Reciprocal Rank Fusion

RRF (Cormack et al., 2009) is the workhorse fusion technique for multi-signal retrieval — used by every search engine that combines BM25 + dense + reranker scores. It avoids the "raw score scales aren't comparable" trap by working on **ranks only**:

```
rrf_score(item) = Σ over runs of  1 / (k + rank_in_run)        # k=60 by convention
```

We fuse on **timestamps**, with a tolerance window (`±5 s` is reasonable — frame timestamp ≠ audio segment start, but they're close).

In [ ]:
from collections import defaultdict

def fuse_rrf(query: str, k: int = 5, top_per_modality: int = 10, k_rrf: int = 60, tol: float = 5.0):
    f_hits = search_frames(query, k=top_per_modality)
    a_hits = search_audio(query,  k=top_per_modality)
    # bucket timestamps to nearest `tol` seconds so frame/audio hits in the same moment merge
    def bucket(ts): return round(ts / tol) * tol
    scores = defaultdict(float)
    sources = defaultdict(set)
    for rank, (ts, _, _) in enumerate(f_hits):
        scores[bucket(ts)] += 1.0 / (k_rrf + rank + 1); sources[bucket(ts)].add("frame")
    for rank, (ts, _, _, _) in enumerate(a_hits):
        scores[bucket(ts)] += 1.0 / (k_rrf + rank + 1); sources[bucket(ts)].add("audio")
    fused = sorted(scores.items(), key=lambda kv: -kv[1])[:k]
    return [(ts, sc, sorted(sources[ts])) for ts, sc in fused]

for q in ["introduction", "main results", "conclusion"]:
    print(f"\n[fused] '{q}':")
    for ts, sc, src in fuse_rrf(q, k=3):
        print(f"  {ts:>5.0f}s  rrf={sc:.4f}  matched: {','.join(src)}")

Notice when **both modalities agree** (`matched: audio,frame`) the RRF score doubles — that's a high-confidence match. When only one signal fires, RRF still surfaces it but with a lower score. This is the "vote of confidence" property that makes RRF the default fusion choice.

## 6. Code-switched query — the bilingual lecture case

If you used a Chinese tech talk, this is where the dual-index approach shines: the audio side is multilingual, the frame side is language-agnostic, and queries can blend languages naturally.

In [ ]:
# Replace these with queries appropriate for your video
MIXED_QUERIES = [
    "解释 attention 是怎么 work 的",   # ZH frame + EN tech terms — typical
    "the chart showing accuracy results",
    "演示代码",
]
for q in MIXED_QUERIES:
    hits = fuse_rrf(q, k=2)
    print(f"\n'{q}'")
    for ts, sc, src in hits:
        print(f"  {ts:>5.0f}s  rrf={sc:.4f}  ({','.join(src)})")

In [ ]:
# Visualize: show the top-1 frame + the audio segment around it
import matplotlib.pyplot as plt

Q = MIXED_QUERIES[0]
top_ts = fuse_rrf(Q, k=1)[0][0]

# nearest sampled frame
frame_idx = min(range(len(frame_seconds)), key=lambda i: abs(frame_seconds[i] - top_ts))
# nearest audio segment
audio_idx = min(range(len(audio_starts)), key=lambda i: abs(audio_starts[i] - top_ts))

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(Image.open(frame_paths[frame_idx]))
ax.set_title(
    f"Query: '{Q}'\nFused timestamp: {top_ts:.0f}s\n"
    f"Audio nearby: {audio_texts[audio_idx][:80]}"
)
ax.set_axis_off(); plt.tight_layout(); plt.show()

## What we built

A working video search engine: input a natural-language query (any language), get the right timestamp(s) in the video. Storage: a few thousand vectors per video.

## Where this would extend in production

- **Persist the indices** in Milvus / Vespa per `video_id`. Pre-compute on upload.
- **Speaker diarization** — WhisperX gives per-speaker segments. Useful for *"what did the panelist named Wendy say about X"*.
- **Scene detection** before frame sampling — `pyscenedetect` cuts frames at scene changes, giving more signal per frame than uniform 1 fps.
- **Cache the rendered frames** keyed by `(video_id, fps)` — re-indexing is the slow part.
- **Smart frame rate by content type** — F-16 fps for sports (§8.1), 1 fps for lectures, scene-cuts for movies.

**Next:** [Notebook 10 — Native vs DIY Video](10_video_native_compare.ipynb). Same video, but sent whole to Gemini's long-context model. When does each architecture win?